# 00. Entorno, corpus y dataset

El experimento compara cuatro estrategias de recuperación y tres generadores sobre preguntas con dos formulaciones. Antes de recalcular las métricas, conviene comprobar qué documentos, fragmentos y preguntas forman el conjunto de análisis.

Este cuaderno lee los materiales incluidos. No descarga documentos ni vuelve a extraer los PDF.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

## Entorno y archivos de entrada

Los scripts se ejecutan con el mismo Python que el kernel. Las rutas siguientes pertenecen al paquete de reproducción; no apuntan a otra versión del TFM.

In [ ]:
import platform
from importlib.metadata import version

display(pd.DataFrame({'componente': ['Python', 'pandas', 'nbclient', 'PyMuPDF'],
                      'version': [platform.python_version(), version('pandas'),
                                  version('nbclient'), version('pymupdf')]}))
paths = ['config/corpus_sources.csv', 'data/processed/chunks_v1.jsonl',
         'data/golden/variants/generation_golden_v3.jsonl',
         'outputs/local_runs/explicitness_v3/qrels_final_v3.jsonl']
assert all((ROOT / path).is_file() for path in paths)
display(pd.DataFrame({'entrada': paths, 'disponible': True}))

## Documentos del corpus

El corpus procede de dos manuales abiertos de OpenStax. El archivo de fuentes conserva la licencia, la URL y los capítulos seleccionados. Los PDF originales se mantienen como referencia de la extracción.

In [ ]:
sources = pd.read_csv(ROOT / 'config/corpus_sources.csv')
assert len(sources) == 2
assert all((ROOT / path).is_file() for path in sources['raw_path'])
display(sources[['title', 'language', 'license', 'included_chapters', 'source_url']])

## Fragmentos utilizados para recuperar evidencia

La segmentación respeta las secciones y utiliza solapamiento. El tamaño se expresa en palabras, no en tokens del generador. El campo formula_risk señala fragmentos cuya extracción puede haber perdido contenido matemático.

In [ ]:
chunks = pd.DataFrame(read_jsonl(ROOT / 'data/processed/chunks_v1.jsonl'))
chunking = json.loads((ROOT / 'data/processed/chunking_report_v1.json').read_text())
assert len(chunks) == chunks['chunk_id'].nunique() == chunking['total_chunks'] == 288
assert chunks['doc_id'].nunique() == 2
display(chunks.groupby(['source_title', 'chapter_id', 'chapter_title'], sort=False)
        .agg(fragmentos=('chunk_id', 'size'), palabras_mediana=('word_count', 'median'))
        .reset_index())

In [ ]:
display(pd.DataFrame([chunking['parameters']]))
display(chunks['word_count'].describe().to_frame('palabras_por_fragmento'))
display(chunks['formula_risk'].value_counts(dropna=False).rename_axis('riesgo_extraccion').to_frame('fragmentos'))

Para comprobar qué llega al recuperador, se puede leer un fragmento completo. Los identificadores conectan este texto con los rankings, las citas y los juicios de relevancia.

In [ ]:
example = chunks.loc[chunks['chunk_id'] == 'openstax_intro_stats_2e:7.1:chunk-001'].iloc[0]
print(example['chunk_id'])
print(example['retrieval_text'])

## Unidades del experimento final

Una pregunta base tiene dos formulaciones. Cada formulación se combina con cuatro recuperadores y tres generadores; las respuestas de una misma base siguen siendo observaciones relacionadas. Los contrastes posteriores agregan por pregunta base.

In [ ]:
golden = pd.DataFrame(read_jsonl(ROOT / 'data/golden/variants/generation_golden_v3.jsonl'))
assert len(golden) == golden['qid'].nunique() == 52
assert golden['base_qid'].nunique() == 26
assert golden.groupby('base_qid').size().eq(2).all()
units = pd.DataFrame({'unidad': ['Preguntas base', 'Formulaciones', 'Recuperadores', 'Generadores', 'Combinaciones previstas'],
                      'n': [golden['base_qid'].nunique(), len(golden), 4, 3, len(golden) * 4 * 3]})
display(units)

El cuaderno 01 examina las preguntas y sus evidencias. Los resultados de recuperación se recalculan en el 02; las respuestas y su evaluación se revisan en el 03.